# Post-Training Analysis — DataGenome Pipeline

Run this notebook after training completes to produce the full sample-level report.

**Pipeline:**
1. Load raw signals saved by `TrainingSignalCollector` during training
2. Compute all 13 derived metrics via `derive_all_metrics()`
3. Categorize each sample into 8 DataGenome categories via `categorize()`
4. Save the final report to disk via `save_categorized_report()`

**Inputs:** `history_metrics/<exp_name>/raw_signals/`  
**Output:** `history_metrics/<exp_name>/sample_map.parquet` + kNN arrays

## 0. Configuration

Set `EXP_NAME` to match the `exp_name` used during training.

In [ ]:
import os
import sys

# Add work_dir to path so nn_tab is importable
sys.path.insert(0, os.path.abspath(".."))

# ── Set experiment name (must match exp_name used during training) ──────────
EXP_NAME = "your_exp_name_here"

# Paths derived automatically from exp_name
RAW_SIGNALS_DIR = os.path.join("..", "history_metrics", EXP_NAME, "raw_signals")
OUTPUT_DIR      = os.path.join("..", "history_metrics", EXP_NAME)

# Hyperparameters (match values used during training if non-default)
K_NEIGHBORS = 10   # kNN density neighbours
K_MCT       = 5    # MCT stability window
E_EARLY     = 5    # EL2N early epochs

print(f"Raw signals dir : {os.path.abspath(RAW_SIGNALS_DIR)}")
print(f"Output dir      : {os.path.abspath(OUTPUT_DIR)}")
print(f"Directory exists: {os.path.isdir(RAW_SIGNALS_DIR)}")

## 1. Load Raw Signals

In [ ]:
import json
import numpy as np

logits_array    = np.load(os.path.join(RAW_SIGNALS_DIR, "logits_array.npy"))
loss_array      = np.load(os.path.join(RAW_SIGNALS_DIR, "loss_array.npy"))
correct_array   = np.load(os.path.join(RAW_SIGNALS_DIR, "correct_array.npy"))
grad_norm_array = np.load(os.path.join(RAW_SIGNALS_DIR, "grad_norm_array.npy"))
embeddings      = np.load(os.path.join(RAW_SIGNALS_DIR, "embeddings.npy"))
labels          = np.load(os.path.join(RAW_SIGNALS_DIR, "targets_array.npy"))

with open(os.path.join(RAW_SIGNALS_DIR, "training_metadata.json")) as f:
    metadata = json.load(f)

N, T, C = logits_array.shape
print(f"Samples (N)     : {N}")
print(f"Epochs  (T)     : {T}")
print(f"Classes (C)     : {C}")
print(f"Embeddings      : {embeddings.shape}")
print(f"Grad norm epochs: {grad_norm_array.shape[1]}")
print(f"\nTraining metadata: {metadata}")

### 1b. Multi-seed logits (optional)

If you ran multiple training seeds, load the final-epoch logits from each seed here.  
Skip this cell and leave `per_seed_final_logits = None` if single-seed.

In [ ]:
# per_seed_final_logits = [
#     np.load("path/to/seed0/logits_array.npy")[:, -1, :],   # [N, C] last epoch
#     np.load("path/to/seed1/logits_array.npy")[:, -1, :],
#     np.load("path/to/seed2/logits_array.npy")[:, -1, :],
# ]

per_seed_final_logits = None   # single-seed run — ensemble_dis will be NaN

## 2. Compute Derived Metrics

Runs all 13 active DataGenome metrics in one call.  
CPU-only, no GPU needed.

In [ ]:
from nn_tab.metrics_generation import derive_all_metrics

metrics_df = derive_all_metrics(
    logits_array          = logits_array,
    loss_array            = loss_array,
    correct_array         = correct_array,
    grad_norm_array       = grad_norm_array,
    embeddings_array      = embeddings,
    labels                = labels,
    k_mct                 = K_MCT,
    e_early               = E_EARLY,
    knn_k                 = K_NEIGHBORS,
    per_seed_final_logits = per_seed_final_logits,
)

print(f"Metrics DataFrame shape: {metrics_df.shape}")
print(f"Columns: {list(metrics_df.columns)}")
metrics_df.head()

In [ ]:
# Quick sanity check — summary stats for all scalar metrics
metrics_df.describe().round(4)

## 3. Categorize Samples

Applies the 8 DataGenome category rules. Categories are **not mutually exclusive** —  
each sample gets a boolean flag per category plus a `primary_category` (first match in priority order).

In [ ]:
from nn_tab.metrics_generation import categorize

report_df = categorize(metrics_df)

print(f"Report DataFrame shape: {report_df.shape}")
print(f"\nPrimary category distribution:")
print(report_df["primary_category"].value_counts())

In [ ]:
# Count of samples per category (a sample can appear in multiple)
category_cols = ["is_noisy", "is_boundary", "is_rare", "is_redundant",
                 "is_easy", "is_mildly_hard", "is_hard", "is_very_hard"]

print("Samples per category (non-exclusive counts):")
for col in category_cols:
    n = report_df[col].sum()
    pct = 100 * n / len(report_df)
    print(f"  {col:<18}: {n:>5}  ({pct:.1f}%)")

In [ ]:
# Samples with more than one category flag (overlap)
report_df["n_categories"] = report_df[category_cols].sum(axis=1)
print("Distribution of how many categories each sample belongs to:")
print(report_df["n_categories"].value_counts().sort_index())

## 4. Save to Disk

In [ ]:
from nn_tab.metrics_generation import save_categorized_report

parquet_path = save_categorized_report(report_df, output_dir=OUTPUT_DIR)

print(f"Saved:")
print(f"  {parquet_path}")
print(f"  {os.path.join(OUTPUT_DIR, 'knn_raw_distances.npy')}")
print(f"  {os.path.join(OUTPUT_DIR, 'knn_normalised_density.npy')}")
print(f"  {os.path.join(OUTPUT_DIR, 'knn_metadata.json')}")

## 5. Quick Inspection

Optional cells to browse the results.

In [ ]:
# View the full report
report_df.drop(columns=["n_categories"]).head(20)

In [ ]:
# Inspect a specific category — e.g. noisy samples
noisy = report_df[report_df["is_noisy"]]
print(f"Noisy samples: {len(noisy)}")
noisy[["label", "aum", "rho", "fe", "ensemble_dis", "primary_category"]].head(10)

In [ ]:
# Inspect rare samples
rare = report_df[report_df["is_rare"]]
print(f"Rare samples: {len(rare)}")
rare[["label", "rho", "knn_density", "prototypicality", "primary_category"]].head(10)